In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from neuralop.models import FNO
import numpy as np

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_dtype(torch.float32)

N_t = 256
N_omega = 10
N_k = 10
T_max = 20.0

t = torch.linspace(0.0, T_max, N_t, device=device)
dt = t[1] - t[0]

In [3]:
omega = torch.linspace(1.0, 10.0, N_omega, device=device)      # (No,)
k = torch.arange(1, N_k + 1, device=device, dtype=torch.float32)  # (Nk,)

t_ = t[:, None, None]                 # (Nt,1,1)
omega_ = omega[None, :, None]         # (1,No,1)
k_ = k[None, None, :]                 # (1,1,Nk)

In [4]:
def mirror_trajectory(t, A, B, kappa):
    z = -t - A * torch.exp(-2 * kappa * t) + B
    dzdt = -1.0 + 2.0 * kappa * A * torch.exp(-2 * kappa * t)
    return z, dzdt

In [5]:
def compute_beta(z, dzdt):
    """
    z, dzdt: (Nt,)
    returns beta: (No, Nk) complex
    """
    z_ = z[:, None, None]              # (Nt,1,1)
    dz_ = dzdt[:, None, None]          # (Nt,1,1)

    phase = torch.exp(
        -1j * omega_ * t_ - 1j * k_ * z_
    )                                  # (Nt,No,Nk)

    prefactor = (-k_ * dz_ - omega_) / torch.sqrt(
        4 * torch.pi * omega_ * 4 * torch.pi * k_
    )

    integrand = phase * prefactor      # (Nt,No,Nk)
    beta = torch.sum(integrand, dim=0) * dt
    return beta                        # (No,Nk) complex

In [6]:
def generate_dataset(n_samples):
    X_list = []
    Y_list = []

    for _ in range(n_samples):
        A = torch.rand(1, device=device) * 0.9 + 0.1
        B = torch.rand(1, device=device) * 0.9 + 0.1
        kappa = torch.rand(1, device=device) * 1.2 + 0.3

        z, dzdt = mirror_trajectory(t, A, B, kappa)
        beta = compute_beta(z, dzdt)   # (No,Nk) complex

        # input channels
        channels = []
        for j in range(N_t):
            channels.append(z[j].expand(N_omega, N_k))

        channels.append(omega[:, None].expand(N_omega, N_k))
        channels.append(k[None, :].expand(N_omega, N_k))

        X = torch.stack(channels, dim=0)          # (C,No,Nk)
        Y = torch.stack([beta.real, beta.imag])   # (2,No,Nk)

        X_list.append(X)
        Y_list.append(Y)

    return (
        torch.stack(X_list),
        torch.stack(Y_list)
    )

In [7]:
X_train, Y_train = generate_dataset(500)
X_test, Y_test = generate_dataset(1)

train_loader = DataLoader(
    TensorDataset(X_train, Y_train),
    batch_size=4,
    shuffle=True
)

In [8]:
model = FNO(
    n_modes=(16, 16),
    hidden_channels=64,
    in_channels=N_t + 2,
    out_channels=2,
    n_layers=4,
    domain_padding=0.1
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

In [9]:
for epoch in range(25):
    model.train()
    total_loss = 0.0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch:02d} | Loss {total_loss/len(train_loader):.3e}")

/home/anshul-jawale/Anshul/Code/Moving-Mirror/env/lib/python3.12/site-packages/tltorch/factorized_tensors/factorized_tensors.py:66: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:306.)
  return self.__class__(self.tensor[indices])
/home/anshul-jawale/Anshul/Code/Moving-Mirror/env/lib/python3.12/site-packages/neuralop/layers/spectral_convolution.py:468: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internal

Epoch 00 | Loss 6.466e-03
Epoch 01 | Loss 3.341e-04
Epoch 02 | Loss 2.962e-04
Epoch 03 | Loss 2.858e-04
Epoch 04 | Loss 2.796e-04
Epoch 05 | Loss 2.614e-04
Epoch 06 | Loss 2.489e-04
Epoch 07 | Loss 2.359e-04
Epoch 08 | Loss 2.283e-04
Epoch 09 | Loss 2.173e-04
Epoch 10 | Loss 2.056e-04
Epoch 11 | Loss 1.914e-04
Epoch 12 | Loss 2.025e-04
Epoch 13 | Loss 1.934e-04
Epoch 14 | Loss 2.078e-04
Epoch 15 | Loss 1.786e-04
Epoch 16 | Loss 1.715e-04
Epoch 17 | Loss 1.672e-04
Epoch 18 | Loss 1.607e-04
Epoch 19 | Loss 1.694e-04
Epoch 20 | Loss 1.633e-04
Epoch 21 | Loss 1.578e-04
Epoch 22 | Loss 1.490e-04
Epoch 23 | Loss 1.525e-04
Epoch 24 | Loss 1.497e-04


In [10]:
#? Testing
model.eval()
with torch.no_grad():
    beta_pred = model(X_test.to(device))[0]
    beta_true = Y_test[0]

beta_pred_c = beta_pred[0] + 1j * beta_pred[1]
beta_true_c = beta_true[0] + 1j * beta_true[1]

In [11]:
(beta_pred_c - beta_true_c)

tensor([[-2.7451e-02+0.0114j, -2.8208e-02+0.0054j, -2.9618e-02-0.0083j,
         -1.8195e-02-0.0118j, -9.1114e-03-0.0180j,  2.2162e-03-0.0077j,
          9.2941e-03+0.0064j,  2.1837e-03+0.0215j, -1.6899e-02+0.0306j,
         -3.6471e-02+0.0266j],
        [-1.9660e-02+0.0207j, -1.9624e-02+0.0198j, -2.4251e-02+0.0120j,
         -2.5194e-02+0.0023j, -2.2977e-02-0.0099j, -1.4658e-02-0.0168j,
         -3.7098e-03-0.0164j,  5.2624e-03-0.0075j,  4.5686e-03+0.0062j,
         -3.6543e-03+0.0170j],
        [-3.8479e-03+0.0212j, -4.6620e-03+0.0222j, -1.0856e-02+0.0213j,
         -1.7184e-02+0.0172j, -2.2919e-02+0.0101j, -2.5299e-02-0.0014j,
         -2.1265e-02-0.0111j, -1.4234e-02-0.0160j, -4.5339e-03-0.0125j,
          2.0050e-03-0.0030j],
        [ 2.2373e-03+0.0068j,  1.6380e-03+0.0076j,  1.6861e-04+0.0129j,
         -2.8298e-03+0.0148j, -1.0033e-02+0.0158j, -1.7147e-02+0.0116j,
         -2.2947e-02+0.0059j, -2.5343e-02-0.0036j, -2.0605e-02-0.0111j,
         -1.1843e-02-0.0136j],
        [-2.

In [12]:
(beta_pred_c - beta_true_c).abs()

tensor([[0.0297, 0.0287, 0.0308, 0.0217, 0.0202, 0.0080, 0.0113, 0.0216, 0.0349,
         0.0451],
        [0.0286, 0.0279, 0.0270, 0.0253, 0.0250, 0.0223, 0.0168, 0.0092, 0.0077,
         0.0174],
        [0.0216, 0.0227, 0.0239, 0.0243, 0.0250, 0.0253, 0.0240, 0.0214, 0.0133,
         0.0036],
        [0.0072, 0.0078, 0.0129, 0.0151, 0.0187, 0.0207, 0.0237, 0.0256, 0.0234,
         0.0180],
        [0.0028, 0.0023, 0.0005, 0.0041, 0.0086, 0.0111, 0.0146, 0.0187, 0.0222,
         0.0217],
        [0.0089, 0.0089, 0.0081, 0.0070, 0.0056, 0.0038, 0.0085, 0.0098, 0.0150,
         0.0196],
        [0.0094, 0.0110, 0.0090, 0.0096, 0.0092, 0.0091, 0.0066, 0.0064, 0.0069,
         0.0122],
        [0.0143, 0.0130, 0.0121, 0.0100, 0.0107, 0.0120, 0.0103, 0.0065, 0.0070,
         0.0073],
        [0.0137, 0.0108, 0.0117, 0.0117, 0.0134, 0.0139, 0.0140, 0.0121, 0.0110,
         0.0056],
        [0.0013, 0.0019, 0.0027, 0.0055, 0.0083, 0.0111, 0.0134, 0.0144, 0.0147,
         0.0124]])

In [13]:
import torch
import torch.special as ts
from scipy.special import loggamma

def beta_davies_correct(omega, p, A, B, kappa):
    """
    omega : (No,) tensor
    p     : (Nk,) tensor
    A,B,kappa : scalars (tensors)
    
    returns beta_davies : (No, Nk) complex tensor
    """

    # reshape for broadcasting
    omega_ = omega[:, None]        # (No,1)
    p_ = p[None, :]                # (1,Nk)

    # D = (1/kappa) ln A - B
    D = (1.0 / kappa) * torch.log(A) - B

    # log of prefactor
    log_pref = (
        -torch.log(torch.tensor(2.0 * torch.pi, device=omega.device))
        -0.5 * (torch.log(p_) + torch.log(omega_))
        - (torch.pi * p_) / (2.0 * kappa)
    )

    # complex phase
    phase = (
        -1j * p_ * D
        +1j * omega_ * B
        -1j * (p_ / kappa) * torch.log(omega_)
    )

    # log-gamma term
    lgamma = loggamma(1.0 + 1j * p_ / kappa)

    log_beta = log_pref + phase + lgamma

    # avoid overflow
    log_beta_real = torch.clamp(log_beta.real, -700.0, 700.0)

    beta = (
        -1j
        * torch.exp(log_beta_real)
        * torch.exp(1j * log_beta.imag)
    )
    print(phase, log_pref, lgamma, log_beta_real)

    return beta  # (No, Nk) complex

In [14]:
#? Beta formula vs Beta Integration
A = torch.tensor(0.6, device=device)
B = torch.tensor(0.3, device=device)
kappa = torch.tensor(0.8, device=device)

z, dzdt = mirror_trajectory(t, A, B, kappa)

# beta from direct integral
beta_int = compute_beta(z, dzdt)          # (No,Nk) complex

# beta from Davies formula
beta_dav = beta_davies_correct(
    omega=omega,
    p=k,
    A=A,
    B=B,
    kappa=kappa
)

# print 10x10 comparison
torch.set_printoptions(precision=4, sci_mode=False)

print("\nBeta from direct integral:\n")
print(beta_int[:10, :10].cpu())

print("\nBeta from Davies formula:\n")
print(beta_dav[:10, :10].cpu())

print("\nAbsolute difference |Δβ|:\n")
print(torch.abs(beta_int - beta_dav)[:10, :10].cpu())


tensor([[0.+1.2385j, 0.+2.1771j, 0.+3.1156j, 0.+4.0541j, 0.+4.9927j, 0.+5.9312j,
         0.+6.8697j, 0.+7.8083j, 0.+8.7468j, 0.+9.6853j],
        [0.+0.6721j, 0.+0.7442j, 0.+0.8163j, 0.+0.8884j, 0.+0.9605j, 0.+1.0326j,
         0.+1.1047j, 0.+1.1768j, 0.+1.2489j, 0.+1.3210j],
        [0.+0.4653j, 0.+0.0305j, 0.-0.4042j, 0.-0.8389j, 0.-1.2737j, 0.-1.7084j,
         0.-2.1431j, 0.-2.5779j, 0.-3.0126j, 0.-3.4473j],
        [0.+0.4057j, 0.-0.3887j, 0.-1.1830j, 0.-1.9773j, 0.-2.7717j, 0.-3.5660j,
         0.-4.3604j, 0.-5.1547j, 0.-5.9490j, 0.-6.7434j],
        [0.+0.4267j, 0.-0.6465j, 0.-1.7198j, 0.-2.7931j, 0.-3.8663j, 0.-4.9396j,
         0.-6.0129j, 0.-7.0861j, 0.-8.1594j, 0.-9.2327j],
        [0.+0.4988j, 0.-0.8023j, 0.-2.1035j, 0.-3.4047j, 0.-4.7058j, 0.-6.0070j,
         0.-7.3082j, 0.-8.6093j, 0.-9.9105j, 0.-11.2117j],
        [0.+0.6061j, 0.-0.8877j, 0.-2.3816j, 0.-3.8754j, 0.-5.3693j, 0.-6.8631j,
         0.-8.3570j, 0.-9.8508j, 0.-11.3447j, 0.-12.8386j],
        [0.+0.7392j, 0.-

/tmp/ipykernel_295575/1955095732.py:36: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  lgamma = loggamma(1.0 + 1j * p_ / kappa)
